<a href="https://colab.research.google.com/github/ETappert/precipitation-downsampling/blob/main/reproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Imports
try:
    import rioxarray as rio
except ImportError:
    !pip install rioxarray
    import rioxarray as rio
import matplotlib.pyplot as plt
from rasterio.warp import calculate_default_transform, reproject, Resampling
import rasterio
import numpy as np

In [4]:
#Prep DEM data
def match_res_prep(match_raster_path):
    with rio.open_rasterio(match_raster_path) as src:
    #clip to the Big Island
      big_island = src.rio.clip_box(minx=-156.07,
                                        miny=18.89,
                                        maxx=-154.799,
                                        maxy=20.277,
                                        )
    #remove nodata values
      nodata_value = big_island.rio.nodata

    #Mask out the nodata values in each band of big_island
      big_island = big_island.where(big_island != nodata_value)

    #Save to raster
      big_island.rio.to_raster('big_island_dem.tif')

    return(big_island)


# Open the precip raster to be resampled
def resample_precip(precip_raster_path, match_raster_path, output_raster='alligned_raster.tif'):
  big_island = match_res_prep(match_raster_path)
  with rasterio.open(precip_raster_path) as src:

    # Open the DEM raster to get its CRS and transform
    with rasterio.open('big_island_dem.tif') as dem:
        dst_crs = dem.crs

        # Calculate new transform for source based on target
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, dem.width, dem.height, *dem.bounds)

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height,
            'count': 4,
            'dtype': rasterio.float32 # Use float32 to accommodate negative elevation values
        })

        # elevation, slope, and aspect channels
        elev = big_island[0].values # Extract NumPy array from xarray DataArray
        slope = big_island[1].values
        aspect = big_island[2].values

        # Reproject and align
        with rasterio.open(output_raster, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest
                )
            # Ensure the data is cast to the correct dtype before writing
            dst.write_band(2, elev.astype(rasterio.float32))
            dst.write_band(3, slope.astype(rasterio.float32))
            dst.write_band(4, aspect.astype(rasterio.float32))


Testing

In [5]:
# @title Mount drive and run imports
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [6]:
data_path = '/content/gdrive/MyDrive/AdvRS/Final/data/3B-HHR-E.MS.MRG.3IMERG.20260320-S173000-E175959.1050.V07C.1day/3B-HHR-E.MS.MRG.3IMERG.20260320-S173000-E175959.1050.V07C.1day.tif'
DEM_path = "/content/gdrive/MyDrive/AdvRS/Final/data/elevation/geostack_30m_topo.tiff"


In [7]:
resample_precip(data_path, DEM_path)